# NoisyNet DQN: learn parameter-space exploration

A noisy layer samples
$$W=\mu_W+\sigma_W\odot\varepsilon_W,\qquad b=\mu_b+\sigma_b\odot\varepsilon_b.$$
Here $\mu$ and $\sigma$ are learned parameters, $\varepsilon$ is factorized Gaussian noise, and $\odot$ is element-wise multiplication. Training resamples noise; evaluation uses only $\mu$.

## 1. Implement factorized noisy linear layers

Factorized noise uses one random vector per input and output, reducing sampling cost from a full random matrix.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

class NoisyLinear(nn.Module):
    def __init__(self, inputs, outputs, sigma=0.5):
        super().__init__()
        self.weight_mu = nn.Parameter(torch.empty(outputs, inputs))
        self.weight_sigma = nn.Parameter(torch.full((outputs, inputs), sigma / np.sqrt(inputs)))
        self.bias_mu = nn.Parameter(torch.empty(outputs))
        self.bias_sigma = nn.Parameter(torch.full((outputs,), sigma / np.sqrt(outputs)))
        nn.init.uniform_(self.weight_mu, -1 / np.sqrt(inputs), 1 / np.sqrt(inputs))
        nn.init.uniform_(self.bias_mu, -1 / np.sqrt(inputs), 1 / np.sqrt(inputs))

    @staticmethod
    def scaled_noise(size):
        noise = torch.randn(size)
        return noise.sign() * noise.abs().sqrt()

    def forward(self, inputs):
        if not self.training:
            return nn.functional.linear(inputs, self.weight_mu, self.bias_mu)
        input_noise = self.scaled_noise(self.weight_mu.shape[1])
        output_noise = self.scaled_noise(self.weight_mu.shape[0])
        weight = self.weight_mu + self.weight_sigma * output_noise.outer(input_noise)
        bias = self.bias_mu + self.bias_sigma * output_noise
        return nn.functional.linear(inputs, weight, bias)

## 2. Build and sample a noisy Q-network

Repeated training-mode passes produce different action values for the same observation without epsilon-greedy action noise.

In [ ]:
network = nn.Sequential(nn.Linear(4, 64), nn.ReLU(), NoisyLinear(64, 2))
observation = torch.tensor([[0.1, -0.2, 0.3, 0.0]])
network.train()
samples = torch.stack([network(observation).squeeze(0) for _ in range(200)]).detach().numpy()
network.eval()
deterministic_values = network(observation).detach().numpy().squeeze(0)
print("Evaluation values:", deterministic_values)

## 3. Visualize learned exploration noise

The histogram shows the distribution of action values induced solely by parameter noise.

In [ ]:
plt.hist(samples[:, 0], bins=25, alpha=0.6, label="Action 0")
plt.hist(samples[:, 1], bins=25, alpha=0.6, label="Action 1")
plt.axvline(deterministic_values[0], color="C0", linestyle="--")
plt.axvline(deterministic_values[1], color="C1", linestyle="--")
plt.xlabel("Sampled Q-value")
plt.ylabel("Count")
plt.title("NoisyNet exploration for one observation")
plt.legend()
plt.show()